# Paper Classification on Colab T4 GPU

End-to-end pipeline: clone repo → install deps → upload data → train 3 tasks → evaluate → export human-review workbook.

**Required runtime:** GPU T4 (Runtime → Change runtime type → T4 GPU).

**Time:**
- Single seed: ~30–45 min for all 3 tasks at 10 epochs each.
- 3-seed ensemble (recommended for max F1): ~90–135 min.

**Cost (free tier OK):** ~15 GPU-hours/week limit; full ensemble run uses ~2.5h.

## 1. Verify GPU

In [ ]:
!nvidia-smi

If you see `Tesla T4` (or `V100`/`A100`) → good. If `command not found` → switch runtime to GPU.

## 2. Clone (or pull) the repo

In [ ]:
import os
if not os.path.exists('paper-classification'):
    !git clone https://github.com/harnetlinh/paper-classification.git
%cd paper-classification
!git pull
!ls

## 3. Install dependencies

In [ ]:
!pip install -q pyarrow openpyxl python-dotenv tenacity

## 4. Upload the input Excel

Upload `2025_11_09_-_biblio_-_du_lieu_phep_phan_tich_-_2013-2024__1_.xlsx` from your machine.

In [ ]:
from google.colab import files
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/2025_11_09_-_biblio_-_du_lieu_phep_phan_tich_-_2013-2024__1_.xlsx'):
    print('Pick the Excel file from your machine:')
    uploaded = files.upload()
    for filename in uploaded.keys():
        os.rename(filename, f'data/{filename}')
!ls -la data/

## 5. Phase 0 — Sanitize

In [ ]:
!python sanitize.py

## 6. Phase 1 — LLM augmentation (optional, skip if augment parquets are in repo)

The repo ships `outputs/{special_edu,ece,tvet,lll}_augmented.parquet` from a previous run.

In [ ]:
# import os; os.environ['OPENAI_API_KEY'] = 'sk-...'
# !python llm_augment.py --class all

## 7. Phase 2 — Train (choose A or B)

**A. Single-seed (fast, ~30–45 min):**

In [ ]:
!python train_specter2.py --task all

**B. 3-seed ensemble (slower, ~90–135 min, +2–5% F1 typical):**

Trains seeds 42, 123, 456 then averages predictions. Use ONLY ONE of A or B.

In [ ]:
# !python train_specter2.py --task all --ensemble

## 8. Phase 3 — Evaluate (val 2023 + test 2024)

Auto-detects ensemble (loads all seed checkpoints if present).

In [ ]:
!python evaluate.py --task all

## 9. Inspect summary

In [ ]:
import json
with open('outputs/eval_report.json') as f:
    rep = json.load(f)
print(f"{'task':<8} | {'macro_f1':>9} {'wgt_f1':>8} {'sup_f1':>8} {'AUC':>6} {'AP':>6}")
print('-' * 60)
for task, r in rep['tasks'].items():
    for split in ['val_2023', 'test_2024']:
        if split in r:
            m = r[split]
            print(f"{task:<8} {split:<10} | "
                  f"{m.get('macro_f1', 0):>9.4f} "
                  f"{m.get('weighted_f1', 0):>8.4f} "
                  f"{m.get('supported_macro_f1', 0):>8.4f} "
                  f"{m.get('macro_auc', 0):>6.3f} "
                  f"{m.get('macro_ap', 0):>6.3f}")

## 10. Export human-review workbook (machine vs human comparison)

Builds `outputs/review.xlsx` — side-by-side machine vs human labels for both val 2023 and test 2024.

**Sheets:**
- `summary_test_2024` / `summary_val_2023`: 1 row/paper, all 3 tasks side-by-side, sorted disagreements-first
- `disagreements_test_2024` / `disagreements_val_2023`: filtered to papers needing human review
- `fields_*` / `levels_*` / `method_*`: per-class probability + status (TP/FP/FN/TN)
- `stats_*`: per-class precision / recall / F1 / support
- `legend`: column glossary

In [ ]:
!python export_review.py --output outputs/review.xlsx
from google.colab import files
files.download('outputs/review.xlsx')

## 11. Download outputs (models + reports)

In [ ]:
!zip -r outputs.zip outputs/*.json outputs/*.pt outputs/*.parquet outputs/*.xlsx 2>/dev/null
from google.colab import files
files.download('outputs.zip')

## 12. (Optional) Inference on a new Scopus file

In [ ]:
# from google.colab import files
# uploaded = files.upload()
# fname = list(uploaded.keys())[0]
# !python inference.py --input {fname} --output predictions.xlsx
# files.download('predictions.xlsx')